# 24 - Integrate Triton Ternary Runtime

Integra o kernel Triton do notebook 23 no runtime nativo do modelo completo.

Objetivos:
- trocar os 78 MLPs por `TritonPackedTernaryLinear`;
- preservar o adapter recovered;
- rodar smoke test e as 100 perguntas por padrão;
- medir load, VRAM/RAM e tokens/s;
- comparar contra os marcos 20 e 22.

Este notebook ainda é `native_triton_prototype`: é integração do kernel no modelo completo, mas não é runtime final/produto.

## Reexecução após a auditoria

Os outputs já gravados abaixo são históricos e foram preservados. As células atualizadas ainda precisam de execução no hardware alvo.

O runtime é extraído integralmente de um único commit: use `RPT_RUNTIME_CHECKOUT` com um checkout Git do RPT; `HEAD` é congelado em um SHA completo. Arquivos locais não commitados não são utilizados. Fora de um checkout, configure `RPT_RUNTIME_REV` com os 40 caracteres do commit publicado; não há download de `main` nem preenchimento parcial de arquivos ausentes. A revisão precisa conter `artifacts.py`, `execution.py` e `metrics.py`.

Os hashes e a revisão são gravados em `runtime_provenance.json`. As medições Triton falham se houver fallback. Essa verificação comprova o backend observado e não comprova ganho de velocidade, generalização ou compatibilidade com outro acelerador. A suíte histórica de 100 perguntas é desenvolvimento, pois foi usada na recuperação. Prewarm nesta sessão não garante cache quente em outra instalação. Instalação de dependências e cópias grandes continuam explícitas nas células históricas; a revisão estática não as executa.


In [ ]:
# Dependências principais.
import os, sys, subprocess, importlib.util, importlib.metadata

required = {
    'torch': 'torch',
    'triton': 'triton',
    'transformers': 'transformers',
    'accelerate': 'accelerate',
    'peft': 'peft',
    'safetensors': 'safetensors',
    'sentencepiece': 'sentencepiece',
    'psutil': 'psutil',
    'packaging': 'packaging',
}
missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    print('Instalando dependências ausentes:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

from packaging.version import Version
def installed_version(package_name):
    try:
        return Version(importlib.metadata.version(package_name))
    except importlib.metadata.PackageNotFoundError:
        return None

torchao_version = installed_version('torchao')
if torchao_version is not None and torchao_version < Version('0.16.0'):
    print(f'Atualizando torchao incompatível: {torchao_version} -> >=0.16.0')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'torchao>=0.16.0'])

print('Dependências prontas.')

In [ ]:
# Bootstrap Colab/local e paths oficiais.
from pathlib import Path
import json, time, gc, shutil, platform

IN_COLAB = 'google.colab' in sys.modules
PROJECT_NAME = 'gemma4_theory'
COLAB_DRIVE_ROOT = '/content/drive/MyDrive/RPT'
LOCAL_ROOT = str(Path.cwd().resolve())

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = Path(COLAB_DRIVE_ROOT) / PROJECT_NAME
else:
    PROJECT_ROOT = Path(LOCAL_ROOT) / PROJECT_NAME if Path(LOCAL_ROOT).name != PROJECT_NAME else Path(LOCAL_ROOT)

CHECKPOINTS_DIR = PROJECT_ROOT / 'checkpoints'
EXPORTS_DIR = PROJECT_ROOT / 'exports'
DEPLOY_DIR = PROJECT_ROOT / 'deploy'
RESULTS_DIR = PROJECT_ROOT / 'results' / 'packed_recovery'
RUNTIME_DIR = PROJECT_ROOT / 'runtime' / 'packed_recovered_l78'
DIST_DIR = PROJECT_ROOT / 'build' / 'distributable_packed_recovered_l78'
DATA_PATH = PROJECT_ROOT / 'data' / 'gemma4_100_questions.json'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DIST_DIR.mkdir(parents=True, exist_ok=True)
if str(RUNTIME_DIR) not in sys.path:
    sys.path.insert(0, str(RUNTIME_DIR))

print('PROJECT_ROOT =', PROJECT_ROOT)
print('RUNTIME_DIR =', RUNTIME_DIR)
print('DIST_DIR =', DIST_DIR)

In [ ]:
# Build with one immutable runtime source; never recover a partial/mixed runtime.
STUDENT_DIR = CHECKPOINTS_DIR / 'student_pruned'
ADAPTER_DIR = CHECKPOINTS_DIR / 'student_packed_ternary_recovered_l78_t4_resume'
PACKED_SOURCE_DIR = next((p for p in [EXPORTS_DIR / 'ternary_mlp_l78', DEPLOY_DIR]
                          if (p / 'manifest.json').exists() and (p / 'mlp_ternary_2bit.bin').exists()), None)

# All runtime files come from one immutable Git revision, never mixed Drive/main files.
import hashlib
import re
import subprocess
import tempfile
import urllib.request

RUNTIME_FILES = [
    'README.md', 'requirements.txt', 'loader.py', 'run_inference.py',
    'native_loader.py', 'run_native_inference.py', 'triton_loader.py',
    'run_triton_inference.py', 'package_manifest.template.json',
    'artifacts.py', 'execution.py', 'metrics.py',
]
# The development evaluator is frozen alongside the runtime, from the same commit.
RUNTIME_SUPPORT_FILES = {'evaluation_policy.py': 'validation/evaluation.py'}
RUNTIME_FILES += list(RUNTIME_SUPPORT_FILES)
RUNTIME_MODULES = ['triton_loader', 'native_loader', 'loader', 'artifacts', 'execution', 'metrics', 'evaluation_policy']
RUNTIME_REPO_PREFIX = 'gemma4_theory/runtime/packed_recovered_l78'
RUNTIME_SOURCE_DIR = None
RUNTIME_PROVENANCE = None

def ensure_runtime_templates():
    global RUNTIME_SOURCE_DIR, RUNTIME_PROVENANCE
    if RUNTIME_SOURCE_DIR is not None:
        for filename, expected in RUNTIME_PROVENANCE['file_sha256'].items():
            if hashlib.sha256((RUNTIME_SOURCE_DIR / filename).read_bytes()).hexdigest() != expected:
                raise RuntimeError(f'Pinned runtime snapshot changed: {filename}')
        return RUNTIME_FILES[:] if RUNTIME_PROVENANCE['source_kind'] == 'github_commit' else []

    checkout = Path(os.environ.get('RPT_RUNTIME_CHECKOUT', str(PROJECT_ROOT.parent))).expanduser().resolve()
    revision = os.environ.get('RPT_RUNTIME_REV', '').strip().lower()
    if revision and re.fullmatch(r'[0-9a-f]{40}', revision) is None:
        raise ValueError('RPT_RUNTIME_REV must be a complete 40-character Git commit SHA')
    try:
        top = subprocess.run(['git', '-C', str(checkout), 'rev-parse', '--show-toplevel'],
                             check=True, capture_output=True, text=True).stdout.strip()
        local_checkout = Path(top).resolve() == checkout
    except (OSError, subprocess.CalledProcessError):
        local_checkout = False

    if local_checkout:
        if not revision:
            revision = subprocess.run(['git', '-C', str(checkout), 'rev-parse', 'HEAD'],
                                      check=True, capture_output=True, text=True).stdout.strip().lower()
        resolved = subprocess.run(['git', '-C', str(checkout), 'rev-parse', '--verify', revision + '^{commit}'],
                                  check=True, capture_output=True, text=True).stdout.strip().lower()
        if resolved != revision or re.fullmatch(r'[0-9a-f]{40}', revision) is None:
            raise RuntimeError('Runtime revision is not an immutable commit')
        source_kind = 'local_git_commit'
    else:
        if not revision:
            raise RuntimeError('Use a local RPT Git checkout or set RPT_RUNTIME_REV to a full published commit SHA. '
                               'Automatic download from main and partial runtime refresh are disabled.')
        source_kind = 'github_commit'

    # Read every file first. An incomplete revision fails before any build is replaced.
    contents = {}
    for filename in RUNTIME_FILES:
        repo_path = RUNTIME_SUPPORT_FILES.get(filename, f'{RUNTIME_REPO_PREFIX}/{filename}')
        if local_checkout:
            try:
                contents[filename] = subprocess.run(['git', '-C', str(checkout), 'show', f'{revision}:{repo_path}'],
                                                    check=True, capture_output=True).stdout
            except subprocess.CalledProcessError as exc:
                raise RuntimeError(f'Runtime revision {revision} is incomplete: {repo_path}') from exc
        else:
            url = f'https://raw.githubusercontent.com/cesarfavero/RPT/{revision}/{repo_path}'
            with urllib.request.urlopen(url, timeout=30) as response:
                contents[filename] = response.read()
    snapshot_parent = PROJECT_ROOT / 'build' / 'runtime_snapshots'
    snapshot_parent.mkdir(parents=True, exist_ok=True)
    snapshot = Path(tempfile.mkdtemp(prefix=revision[:12] + '-', dir=snapshot_parent))
    try:
        for filename, content in contents.items():
            (snapshot / filename).write_bytes(content)
        provenance = {'source_kind': source_kind, 'git_revision': revision,
                      'working_tree_used': False,
                      'file_sha256': {name: hashlib.sha256(data).hexdigest() for name, data in contents.items()}}
        (snapshot / 'runtime_provenance.json').write_text(json.dumps(provenance, indent=2) + '\n', encoding='utf-8')
    except BaseException:
        shutil.rmtree(snapshot, ignore_errors=True)
        raise
    RUNTIME_SOURCE_DIR, RUNTIME_PROVENANCE = snapshot, provenance
    return RUNTIME_FILES[:] if source_kind == 'github_commit' else []

def activate_runtime_directory(runtime_dir):
    runtime_dir = Path(runtime_dir).resolve()
    provenance = json.loads((runtime_dir / 'runtime_provenance.json').read_text(encoding='utf-8'))
    if provenance != RUNTIME_PROVENANCE:
        raise RuntimeError('Bundle runtime revision differs from the frozen notebook runtime')
    for filename, expected in provenance['file_sha256'].items():
        if hashlib.sha256((runtime_dir / filename).read_bytes()).hexdigest() != expected:
            raise RuntimeError(f'Bundle runtime file hash mismatch: {filename}')
    runtime_str = str(runtime_dir)
    if runtime_str in sys.path:
        sys.path.remove(runtime_str)
    sys.path.insert(0, runtime_str)
    for module_name in RUNTIME_MODULES:
        sys.modules.pop(module_name, None)
    import importlib
    importlib.invalidate_caches()

def copytree_clean(src, dst):
    src, dst = Path(src), Path(dst)
    if dst.exists() or dst.is_symlink():
        if dst.is_symlink() or dst.is_file():
            dst.unlink()
        else:
            shutil.rmtree(dst)
    shutil.copytree(src, dst, ignore=shutil.ignore_patterns('__pycache__', '*.pyc'))

def copy_file_clean(src, dst):
    src, dst = Path(src), Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    shutil.copy2(src, dst)

def ensure_distributable():
    if PACKED_SOURCE_DIR is None:
        raise FileNotFoundError('Packed source not found in exports/ternary_mlp_l78 or deploy')
    downloaded = ensure_runtime_templates()
    required_sources = [STUDENT_DIR, ADAPTER_DIR, RUNTIME_SOURCE_DIR]
    missing = [str(path) for path in required_sources if not path.exists()]
    if missing:
        raise FileNotFoundError('Missing sources: ' + ', '.join(missing))
    for src, dst in [(STUDENT_DIR, DIST_DIR / 'student_pruned'),
                     (ADAPTER_DIR, DIST_DIR / 'adapter_recovered_l78'),
                     (RUNTIME_SOURCE_DIR, DIST_DIR / 'runtime')]:
        copytree_clean(src, dst)
    (DIST_DIR / 'packed').mkdir(parents=True, exist_ok=True)
    copy_file_clean(PACKED_SOURCE_DIR / 'manifest.json', DIST_DIR / 'packed' / 'manifest.json')
    copy_file_clean(PACKED_SOURCE_DIR / 'mlp_ternary_2bit.bin', DIST_DIR / 'packed' / 'mlp_ternary_2bit.bin')
    # Metadata from a previous build does not certify the rebuilt model.
    for stale in ('artifact_checksums.json', 'production_manifest.json', 'prewarm_profile.json', 'package_manifest.json'):
        (DIST_DIR / stale).unlink(missing_ok=True)
    (DIST_DIR / 'README.md').write_text('RPT packed runtime build\nRuntime revision: ' + RUNTIME_PROVENANCE['git_revision'] + '\nQuality has not been revalidated by packaging. The historical 100-question suite is development data.\n', encoding='utf-8')
    return downloaded

runtime_downloaded = ensure_distributable()
runtime_dist = DIST_DIR / 'runtime'
activate_runtime_directory(runtime_dist)
print('distributable ready:', DIST_DIR)
print('runtime revision:', RUNTIME_PROVENANCE['git_revision'])
print('runtime files:', RUNTIME_FILES)

def synchronize_model_devices(model):
    # PyTorch may expose non-NVIDIA devices through the CUDA interface.
    # Synchronize the actual tensor devices, not only the current device.
    from itertools import chain
    devices = {tensor.device for tensor in chain(model.parameters(), model.buffers())
               if tensor.device.type == 'cuda'}
    for device in sorted(devices, key=str):
        torch.cuda.synchronize(device)


In [ ]:
# Carrega runtime Triton e mede load/memória.
import psutil
import torch
import triton
from triton_loader import load_triton_packed_recovered_l78, runtime_execution_report, reset_runtime_execution

assert torch.cuda.is_available(), 'Runtime Triton precisa de CUDA.'
process = psutil.Process(os.getpid())
HF_TOKEN = os.environ.get('HF_TOKEN')
BLOCK_M = int(os.environ.get('RPT_TRITON_BLOCK_M', '16'))
BLOCK_N = int(os.environ.get('RPT_TRITON_BLOCK_N', '16'))
BLOCK_K = int(os.environ.get('RPT_TRITON_BLOCK_K', '64'))

def format_bytes(n):
    units = ['B', 'KB', 'MB', 'GB', 'TB']
    value = float(n)
    for unit in units:
        if value < 1024 or unit == units[-1]:
            return f'{value:.2f} {unit}'
        value /= 1024

def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def memory_snapshot(label):
    snap = {'label': label, 'rss_bytes': process.memory_info().rss, 'rss_human': format_bytes(process.memory_info().rss)}
    if torch.cuda.is_available():
        snap.update({
            'cuda_device': torch.cuda.get_device_name(0),
            'cuda_allocated_bytes': torch.cuda.memory_allocated(),
            'cuda_reserved_bytes': torch.cuda.memory_reserved(),
            'cuda_max_allocated_bytes': torch.cuda.max_memory_allocated(),
            'cuda_allocated_human': format_bytes(torch.cuda.memory_allocated()),
            'cuda_reserved_human': format_bytes(torch.cuda.memory_reserved()),
            'cuda_max_allocated_human': format_bytes(torch.cuda.max_memory_allocated()),
        })
    return snap

cleanup_memory()
torch.cuda.reset_peak_memory_stats()
before_load = memory_snapshot('before_triton_load')
started = time.perf_counter()
model, tokenizer, triton_report = load_triton_packed_recovered_l78(
    DIST_DIR,
    token=HF_TOKEN,
    device_map='auto',
    dtype='auto',
    block_m=BLOCK_M,
    block_n=BLOCK_N,
    block_k=BLOCK_K,
    verify_sha=True, require_triton=True, fallback_to_pytorch=False,
)
load_elapsed = time.perf_counter() - started
after_load = memory_snapshot('after_triton_load')
print(json.dumps({'load_elapsed_sec': load_elapsed, 'before': before_load, 'after': after_load, 'triton_report': triton_report}, ensure_ascii=False, indent=2))
reset_runtime_execution(model)


In [ ]:
# Smoke test e avaliação 100 perguntas por padrão.
PROMPT_TEMPLATE = 'Responda de forma curta e direta.\nPergunta: {question}\nResposta:'
DEFAULT_PROMPTS = [
    'Qual a capital da França?',
    'Quanto é 12 x 8?',
    'Quem desenvolveu a teoria da relatividade?',
    'Em Python, qual palavra-chave define uma função?',
    'Se hoje é segunda-feira, que dia será depois de 3 dias?',
]

def generate_answer(question, max_new_tokens=32):
    prompt = PROMPT_TEMPLATE.format(question=question)
    inputs = tokenizer(prompt, return_tensors='pt')
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    synchronize_model_devices(model)
    started = time.perf_counter()
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    synchronize_model_devices(model)
    elapsed = time.perf_counter() - started
    prompt_tokens = int(inputs['input_ids'].shape[-1])
    generated_tokens = int(output.shape[-1]) - prompt_tokens
    answer = tokenizer.decode(output[0][prompt_tokens:], skip_special_tokens=True).strip()
    return {
        'question': question,
        'answer': answer,
        'prompt_tokens': prompt_tokens,
        'generated_tokens': generated_tokens,
        'elapsed_sec': elapsed,
        'tokens_per_sec': generated_tokens / elapsed if elapsed > 0 else None,
    }

def accepted_answers_for_item(item):
    answers = []
    for key in ('accepted_answers', 'answers'):
        value = item.get(key)
        if isinstance(value, list):
            answers.extend(str(v) for v in value)
    for key in ('expected_answer', 'answer', 'expected'):
        value = item.get(key)
        if value is not None:
            answers.append(str(value))
    answers = [a.strip() for a in answers if str(a).strip()]
    if not answers:
        raise KeyError(f'Nenhum gabarito encontrado no item: {item}')
    return list(dict.fromkeys(answers))

# Use the shared evaluator frozen from the same Git revision as the runtime.
from evaluation_policy import EVALUATOR_VERSION, answer_is_correct

def answer_matches(got, accepted):
    return answer_is_correct(got, accepted, method='exact')

smoke_rows = [generate_answer(q) for q in DEFAULT_PROMPTS]
for row in smoke_rows:
    print(json.dumps(row, ensure_ascii=False))

RUN_FULL_100 = os.environ.get('RPT_TRITON_RUN_FULL_100', '1') == '1'
eval_summary = None
eval_rows = []
if RUN_FULL_100:
    questions = json.loads(DATA_PATH.read_text(encoding='utf-8'))
    correct = 0
    by_category = {}
    for idx, item in enumerate(questions, 1):
        row = generate_answer(item['question'], max_new_tokens=24)
        accepted = accepted_answers_for_item(item)
        ok = answer_matches(row['answer'], accepted)
        correct += int(ok)
        category = item.get('category', 'unknown')
        cat = by_category.setdefault(category, {'total': 0, 'correct': 0})
        cat['total'] += 1
        cat['correct'] += int(ok)
        row.update({'expected': accepted[0], 'accepted_answers': accepted, 'ok': ok, 'category': category})
        eval_rows.append(row)
        print(f"{idx:03d}/{len(questions)} {'OK' if ok else 'FAIL'} | {item['question']} => {row['answer']}")
    for cat in by_category.values():
        cat['accuracy'] = cat['correct'] / cat['total']
    eval_summary = {'classification': 'development', 'evaluator_version': EVALUATOR_VERSION, 'evaluator_sha256': RUNTIME_PROVENANCE['file_sha256']['evaluation_policy.py'], 'training_exposure': True, 'total_questions': len(questions), 'correct': correct, 'accuracy': correct / len(questions), 'by_category': by_category}
else:
    print('Avaliação 100 perguntas pulada por RPT_TRITON_RUN_FULL_100=0.')

In [ ]:
# Salva evidência observada após a execução.
synchronize_model_devices(model)
triton_report.update(runtime_execution_report(model, require_triton=True))
smoke_summary = {
    'num_prompts': len(smoke_rows),
    'total_generated_tokens': sum(r['generated_tokens'] for r in smoke_rows),
    'total_elapsed_sec': sum(r['elapsed_sec'] for r in smoke_rows),
    'aggregate_tokens_per_sec': sum(r['generated_tokens'] for r in smoke_rows) / sum(r['elapsed_sec'] for r in smoke_rows),
}

result = {
    'notebook': '24_integrate_triton_ternary_runtime',
    'purpose': 'Full model integration of Triton packed ternary MLP runtime.',
    'created_at': time.strftime('%Y-%m-%d %H:%M:%S'),
    'runtime_kind': 'native_triton_prototype',
    'is_native_ternary_runtime': triton_report['packed_triton_execution_verified'],
    'is_optimized_kernel': None,
    'performance_validated': False,
    'runtime_source': RUNTIME_PROVENANCE,
    'evaluation_classification': 'development',
    'load_elapsed_sec': load_elapsed,
    'memory_before_load': before_load,
    'memory_after_load': after_load,
    'triton_report': triton_report,
    'smoke_rows': smoke_rows,
    'smoke_summary': smoke_summary,
    'eval_summary': eval_summary,
    'eval_rows': eval_rows,
    'baseline_context': {
        'notebook_20_materialized_tokens_per_sec': 7.746480522791005,
        'notebook_20_materialized_cuda_allocated_after_load': '5.10 GB',
        'notebook_22_native_structural_tokens_per_sec': 2.285913541069469,
        'notebook_22_native_structural_cuda_allocated_after_load': '2.40 GB',
        'notebook_23_triton_spike_speedup_vs_unpack': 187.42602802943765,
    },
    'next_step': 'Evaluate held-out external tasks on the exact packed artifact before making quality or speed claims.',
}

out_path = RESULTS_DIR / f"24_integrate_triton_ternary_runtime_{time.strftime('%Y%m%d_%H%M%S')}.json"
out_path.write_text(json.dumps(result, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print('saved:', out_path)
print(json.dumps({'smoke_summary': smoke_summary, 'eval_summary': eval_summary}, ensure_ascii=False, indent=2))

## Critério de sucesso

O 24 só é considerado positivo se:

1. `triton_report.replaced_count == 78`;
2. `eval_summary.accuracy == 1.0` no benchmark local de 100 perguntas;
3. VRAM alocada após load continuar próxima do runtime nativo estrutural (`~2.40 GB`);
4. tokens/s superar claramente o notebook 22 (`~2.29 tokens/s`).

Se preservar qualidade mas não melhorar velocidade no modelo completo, o próximo passo não é integração final; é tuning de kernel/blocking e profiling por camada.